In [ ]:
# ============================================================
# DAY 09 - SIRAH RAG
# STEP 1 - FRESH EXTRACTION FROM ORIGINAL PDF
#
# PyMuPDF only.
# Reconstructs missing Arabic word spaces using PDF geometry.
# ============================================================

import os
import json
import re
import shutil
import unicodedata
import statistics
import pymupdf


# ============================================================
# PATHS
# ============================================================

PDF_PATH = r"\\192.168.1.10\shear all\على هامش السيرة_Foulabook.com_.pdf"

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_sirah_rag"


# ============================================================
# RESET ENTIRE PROJECT
# ============================================================

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)


# ============================================================
# CREATE PROJECT STRUCTURE
# ============================================================

folders = [
    "01_extracted",
    "02_cleaned",
    "03_chunks",
    "04_embeddings",
    "05_vector_store",
    "06_results"
]

for folder in folders:
    os.makedirs(
        os.path.join(
            PROJECT_DIR,
            folder
        ),
        exist_ok=True
    )


OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "01_extracted"
)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "sirah_extracted.json"
)

OUTPUT_TXT = os.path.join(
    OUTPUT_DIR,
    "sirah_extracted.txt"
)


# ============================================================
# PDF CHECK
# ============================================================

print("=" * 70)
print("FRESH SIRAH PDF EXTRACTION")
print("=" * 70)

print(
    f"\nPDF:\n{PDF_PATH}"
)

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PDF not found:\n{PDF_PATH}"
    )


# ============================================================
# OPEN PDF
# ============================================================

print(
    "\nOpening PDF..."
)

doc = pymupdf.open(
    PDF_PATH
)

print(
    f"PDF pages: {len(doc)}"
)


# ============================================================
# UNICODE NORMALIZATION
# ============================================================

BIDI_CONTROLS = re.compile(
    r"[\u202A-\u202E\u2066-\u2069]"
)

ZERO_WIDTH = re.compile(
    r"[\u200B\u200C\u200D\uFEFF]"
)


def normalize_char(text):

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = BIDI_CONTROLS.sub(
        "",
        text
    )

    text = ZERO_WIDTH.sub(
        "",
        text
    )

    return text


def contains_combining(text):

    return any(
        unicodedata.category(ch)
        in ("Mn", "Mc", "Me")
        for ch in text
    )


def is_whitespace(text):

    return text.isspace()


# ============================================================
# RECONSTRUCT ONE LINE
# ============================================================

def reconstruct_line(line):

    chars = []

    for span in line.get(
        "spans",
        []
    ):

        for char in span.get(
            "chars",
            []
        ):

            value = char.get(
                "c",
                ""
            )

            if value:
                chars.append(
                    {
                        "text": normalize_char(value),
                        "bbox": char["bbox"]
                    }
                )

    if not chars:
        return ""


    # --------------------------------------------------------
    # Detect visual reading direction
    # --------------------------------------------------------

    x_positions = [
        item["bbox"][0]
        for item in chars
    ]

    rtl = (
        len(x_positions) > 1
        and x_positions[0]
        > x_positions[-1]
    )


    # --------------------------------------------------------
    # For Arabic, PyMuPDF stores the visual sequence
    # from right to left. For Latin, left to right.
    # --------------------------------------------------------

    if rtl:

        chars = sorted(
            chars,
            key=lambda item:
                item["bbox"][0],
            reverse=True
        )

    else:

        chars = sorted(
            chars,
            key=lambda item:
                item["bbox"][0]
        )


    # --------------------------------------------------------
    # Estimate normal character height
    # --------------------------------------------------------

    heights = [
        item["bbox"][3]
        - item["bbox"][1]
        for item in chars
        if item["bbox"][3]
        > item["bbox"][1]
    ]

    if heights:

        median_height = statistics.median(
            heights
        )

    else:

        median_height = 10.0


    # --------------------------------------------------------
    # Missing PDF word spaces are much larger than
    # normal intra-word character gaps.
    # --------------------------------------------------------

    SPACE_THRESHOLD = max(
        1.2,
        median_height * 0.15
    )


    output = []


    for i, current in enumerate(chars):

        current_text = current["text"]


        if i > 0:

            previous = chars[i - 1]

            previous_text = previous["text"]


            # ------------------------------------------------
            # Calculate visual gap
            # ------------------------------------------------

            if rtl:

                gap = (
                    previous["bbox"][0]
                    -
                    current["bbox"][2]
                )

            else:

                gap = (
                    current["bbox"][0]
                    -
                    previous["bbox"][2]
                )


            # ------------------------------------------------
            # Don't insert spaces inside combining marks
            # ------------------------------------------------

            combining_nearby = (
                contains_combining(
                    previous_text
                )
                or
                contains_combining(
                    current_text
                )
            )


            # ------------------------------------------------
            # Don't insert spaces around punctuation
            # ------------------------------------------------

            punctuation_nearby = (
                previous_text in "،؛.!?:؟»)]}"
                or
                current_text in "،؛.!?:؟»)]}"
            )


            # ------------------------------------------------
            # Insert missing visual word space
            # ------------------------------------------------

            if (
                gap > SPACE_THRESHOLD
                and not combining_nearby
                and not punctuation_nearby
            ):

                if (
                    output
                    and
                    output[-1] != " "
                ):

                    output.append(
                        " "
                    )


        output.append(
            current_text
        )


    return "".join(
        output
    ).strip()


# ============================================================
# EXTRACT ONE PAGE
# ============================================================

def extract_page(page):

    raw_dict = page.get_text(
        "rawdict",
        sort=False
    )

    page_lines = []


    for block in raw_dict.get(
        "blocks",
        []
    ):

        if "lines" not in block:
            continue


        for line in block["lines"]:

            text = reconstruct_line(
                line
            )


            if text:

                page_lines.append(
                    text
                )


    return "\n".join(
        page_lines
    ).strip()


# ============================================================
# FINAL PAGE NORMALIZATION
# ============================================================

def normalize_page_text(text):

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = BIDI_CONTROLS.sub(
        "",
        text
    )

    text = ZERO_WIDTH.sub(
        "",
        text
    )

    text = text.replace(
        "\r\n",
        "\n"
    )

    text = text.replace(
        "\r",
        "\n"
    )

    lines = []

    for line in text.split("\n"):

        line = line.strip()

        if line:

            lines.append(
                line
            )

    return "\n".join(
        lines
    ).strip()


# ============================================================
# EXTRACT ALL PAGES
# ============================================================

pages_data = []

print(
    "\nExtracting PDF..."
)

for index, page in enumerate(doc):

    page_number = index + 1


    text = extract_page(
        page
    )


    text = normalize_page_text(
        text
    )


    pages_data.append(
        {
            "page":
                page_number,

            "text":
                text,

            "characters":
                len(text),

            "words":
                len(text.split())
        }
    )


    if page_number % 25 == 0:

        print(
            f"Processed "
            f"{page_number}/"
            f"{len(doc)} pages"
        )


doc.close()


# ============================================================
# SAVE JSON
# ============================================================

document_data = {

    "document":
        "على هامش السيرة",

    "source_pdf":
        PDF_PATH,

    "extraction_method":
        "PyMuPDF geometry-based Arabic extraction",

    "ocr_used":
        False,

    "total_pdf_pages":
        len(pages_data),

    "pages_data":
        pages_data
}


with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        document_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SAVE TXT
# ============================================================

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    for page in pages_data:

        f.write(
            "=" * 70
        )

        f.write("\n")

        f.write(
            f"PAGE {page['page']}"
        )

        f.write("\n")

        f.write(
            "=" * 70
        )

        f.write("\n\n")

        f.write(
            page["text"]
        )

        f.write("\n\n")


# ============================================================
# PREVIEW
# ============================================================

print("\n")

print("=" * 70)
print("PREVIEW")
print("=" * 70)


preview_pages = [
    4,
    5,
    6,
    8,
    10,
    16,
    407
]


for page in pages_data:

    if page["page"] in preview_pages:

        print("\n")

        print(
            f"PAGE {page['page']}"
        )

        print(
            "-" * 70
        )

        print(
            page["text"][:3000]
        )


# ============================================================
# STATISTICS
# ============================================================

total_characters = sum(
    page["characters"]
    for page in pages_data
)

total_words = sum(
    page["words"]
    for page in pages_data
)


print("\n")

print("=" * 70)
print("EXTRACTION COMPLETE")
print("=" * 70)

print(
    f"\nPages          : "
    f"{len(pages_data)}"
)

print(
    f"Characters     : "
    f"{total_characters:,}"
)

print(
    f"Words          : "
    f"{total_words:,}"
)

print(
    "\nJSON saved to:"
)

print(
    OUTPUT_JSON
)

print(
    "\nTXT saved to:"
)

print(
    OUTPUT_TXT
)

In [ ]:
# ============================================================
# DAY 09 - SIRAH RAG
# STEP 2 - CHUNKING
# ============================================================

import os
import json
import re


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_sirah_rag"

INPUT_JSON = os.path.join(
    PROJECT_DIR,
    "01_extracted",
    "sirah_extracted.json"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "03_chunks"
)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "sirah_chunks.json"
)

OUTPUT_TXT = os.path.join(
    OUTPUT_DIR,
    "sirah_chunks.txt"
)


# ============================================================
# SETTINGS
# ============================================================

CHUNK_SIZE = 150
CHUNK_OVERLAP = 50

STEP_SIZE = (
    CHUNK_SIZE
    - CHUNK_OVERLAP
)


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# LOAD EXTRACTION
# ============================================================

with open(
    INPUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    document_data = json.load(f)


pages_data = document_data["pages_data"]


print("=" * 70)
print("SIRAH TEXT CHUNKING")
print("=" * 70)

print(
    f"\nPages         : {len(pages_data):,}"
)

print(
    f"Chunk size    : {CHUNK_SIZE}"
)

print(
    f"Overlap       : {CHUNK_OVERLAP}"
)

print(
    f"Step          : {STEP_SIZE}"
)


# ============================================================
# CLEAN TEXT
# ============================================================

def clean_text(text):

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ============================================================
# CREATE CHUNKS
# ============================================================

chunks = []

chunk_id = 0


for page_data in pages_data:

    page_number = page_data["page"]

    text = clean_text(
        page_data["text"]
    )

    if not text:
        continue

    words = text.split()

    if not words:
        continue

    start = 0


    while start < len(words):

        end = min(
            start + CHUNK_SIZE,
            len(words)
        )

        chunk_words = words[
            start:end
        ]

        chunk_text = " ".join(
            chunk_words
        ).strip()


        if chunk_text:

            chunk_id += 1

            chunks.append(
                {
                    "chunk_id":
                        f"chunk_{chunk_id:04d}",

                    "page_start":
                        page_number,

                    "page_end":
                        page_number,

                    "word_count":
                        len(chunk_words),

                    "text":
                        chunk_text
                }
            )


        if end >= len(words):
            break


        start += STEP_SIZE


# ============================================================
# BUILD OUTPUT
# ============================================================

chunks_data = {

    "document":
        document_data["document"],

    "source_pdf":
        document_data["source_pdf"],

    "chunking_method":
        "fixed word-size chunks with overlap",

    "chunk_size":
        CHUNK_SIZE,

    "chunk_overlap":
        CHUNK_OVERLAP,

    "step_size":
        STEP_SIZE,

    "total_chunks":
        len(chunks),

    "chunks":
        chunks
}


# ============================================================
# SAVE JSON
# ============================================================

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        chunks_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# SAVE TXT
# ============================================================

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    for chunk in chunks:

        f.write(
            "=" * 70
        )

        f.write("\n")

        f.write(
            f"{chunk['chunk_id']}\n"
        )

        f.write(
            f"Pages: "
            f"{chunk['page_start']}-"
            f"{chunk['page_end']}\n"
        )

        f.write(
            f"Words: "
            f"{chunk['word_count']}\n\n"
        )

        f.write(
            chunk["text"]
        )

        f.write("\n\n")


# ============================================================
# STATISTICS
# ============================================================

word_counts = [
    chunk["word_count"]
    for chunk in chunks
]

total_words = sum(
    word_counts
)

average_words = (
    total_words / len(chunks)
    if chunks
    else 0
)

minimum_words = (
    min(word_counts)
    if chunks
    else 0
)

maximum_words = (
    max(word_counts)
    if chunks
    else 0
)


# ============================================================
# PREVIEW
# ============================================================

print("\n")
print("=" * 70)
print("CHUNK PREVIEW")
print("=" * 70)


for chunk in chunks[:5]:

    print("\n")

    print(
        f"{chunk['chunk_id']} | "
        f"Page {chunk['page_start']} | "
        f"{chunk['word_count']} words"
    )

    print("-" * 70)

    print(
        chunk["text"][:1000]
    )


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 70)
print("CHUNKING COMPLETE")
print("=" * 70)

print(
    f"\nTotal chunks : {len(chunks):,}"
)

print(
    f"Total words  : {total_words:,}"
)

print(
    f"Average      : {average_words:.1f}"
)

print(
    f"Minimum      : {minimum_words}"
)

print(
    f"Maximum      : {maximum_words}"
)

print(
    f"\nJSON:\n{OUTPUT_JSON}"
)

print(
    f"\nTXT:\n{OUTPUT_TXT}"
)

In [ ]:
# ============================================================
# DAY 09 - SIRAH RAG
# STEP 3 - ARABIC BGE-M3 EMBEDDINGS
# ============================================================

import os
import json
import numpy as np

from sentence_transformers import SentenceTransformer


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_sirah_rag"

INPUT_JSON = os.path.join(
    PROJECT_DIR,
    "03_chunks",
    "sirah_chunks.json"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "04_embeddings"
)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "sirah_embeddings.json"
)


# ============================================================
# SETTINGS
# ============================================================

MODEL_NAME = (
    "sayed0am/arabic-english-bge-m3"
)

BATCH_SIZE = 16


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# LOAD CHUNKS
# ============================================================

with open(
    INPUT_JSON,
    "r",
    encoding="utf-8"
) as f:

    chunks_data = json.load(f)


chunks = chunks_data["chunks"]

texts = [
    chunk["text"]
    for chunk in chunks
]


print("=" * 70)
print("ARABIC BGE-M3 EMBEDDINGS")
print("=" * 70)

print(
    f"\nChunks : {len(chunks):,}"
)

print(
    f"Model  : {MODEL_NAME}"
)


# ============================================================
# LOAD MODEL
# ============================================================

print(
    "\nLoading embedding model..."
)

model = SentenceTransformer(
    MODEL_NAME
)


# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

print(
    "\nGenerating embeddings..."
)

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
)


# ============================================================
# NUMPY
# ============================================================

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)


# ============================================================
# VERIFY
# ============================================================

if len(embeddings) != len(chunks):

    raise ValueError(
        "Embedding count does not match chunk count."
    )


print(
    f"\nEmbedding shape: "
    f"{embeddings.shape}"
)


# ============================================================
# SAVE
# ============================================================

embedding_data = {

    "document":
        chunks_data["document"],

    "source_pdf":
        chunks_data["source_pdf"],

    "embedding_model":
        MODEL_NAME,

    "dimensions":
        int(embeddings.shape[1]),

    "normalized":
        True,

    "total_embeddings":
        int(len(embeddings)),

    "embeddings":
        embeddings.tolist()
}


with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        embedding_data,
        f,
        ensure_ascii=False
    )


# ============================================================
# FINAL
# ============================================================

file_size = (
    os.path.getsize(
        OUTPUT_JSON
    )
    / (1024 * 1024)
)


print("\n")
print("=" * 70)
print("EMBEDDINGS COMPLETE")
print("=" * 70)

print(
    f"\nEmbeddings : "
    f"{len(embeddings):,}"
)

print(
    f"Dimensions : "
    f"{embeddings.shape[1]}"
)

print(
    f"File size  : "
    f"{file_size:.2f} MB"
)

print(
    f"\nSaved to:\n{OUTPUT_JSON}"
)

In [ ]:
# ============================================================
# DAY 09 - SIRAH RAG
# STEP 4 - BUILD FAISS VECTOR STORE
# ============================================================

import os
import json
import numpy as np
import faiss


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_sirah_rag"

EMBEDDINGS_JSON = os.path.join(
    PROJECT_DIR,
    "04_embeddings",
    "sirah_embeddings.json"
)

CHUNKS_JSON = os.path.join(
    PROJECT_DIR,
    "03_chunks",
    "sirah_chunks.json"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "05_vector_store"
)

INDEX_PATH = os.path.join(
    OUTPUT_DIR,
    "sirah_faiss.index"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "sirah_metadata.json"
)


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# LOAD EMBEDDINGS
# ============================================================

print("=" * 70)
print("BUILDING FAISS VECTOR STORE")
print("=" * 70)

with open(
    EMBEDDINGS_JSON,
    "r",
    encoding="utf-8"
) as f:

    embedding_data = json.load(f)


embeddings = np.asarray(
    embedding_data["embeddings"],
    dtype=np.float32
)


# ============================================================
# LOAD CHUNKS
# ============================================================

with open(
    CHUNKS_JSON,
    "r",
    encoding="utf-8"
) as f:

    chunks_data = json.load(f)


chunks = chunks_data["chunks"]


# ============================================================
# VERIFY
# ============================================================

print(
    f"\nEmbeddings : {len(embeddings):,}"
)

print(
    f"Chunks     : {len(chunks):,}"
)

if len(embeddings) != len(chunks):

    raise ValueError(
        "Embeddings and chunks do not match."
    )


# ============================================================
# BUILD INDEX
# ============================================================

dimension = embeddings.shape[1]


print(
    f"Dimension   : {dimension}"
)

print(
    "\nBuilding FAISS..."
)


index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)


# ============================================================
# SAVE INDEX
# ============================================================

faiss.write_index(
    index,
    INDEX_PATH
)


# ============================================================
# SAVE METADATA
# ============================================================

metadata = {

    "document":
        chunks_data["document"],

    "source_pdf":
        chunks_data["source_pdf"],

    "embedding_model":
        embedding_data["embedding_model"],

    "dimensions":
        dimension,

    "total_vectors":
        int(index.ntotal),

    "similarity":
        "cosine similarity using normalized vectors",

    "chunks":
        chunks
}


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# VERIFY
# ============================================================

test_index = faiss.read_index(
    INDEX_PATH
)


print("\n")
print("=" * 70)
print("FAISS COMPLETE")
print("=" * 70)

print(
    f"\nVectors    : "
    f"{test_index.ntotal:,}"
)

print(
    f"Dimensions : "
    f"{test_index.d}"
)

print(
    f"\nIndex:\n{INDEX_PATH}"
)

print(
    f"\nMetadata:\n{METADATA_PATH}"
)

In [ ]:
# ============================================================
# DAY 09 - SIRAH RAG
# STEP 5 - HYBRID RETRIEVAL + RERANKING
# ============================================================

import os
import json
import re
import math

import numpy as np
import faiss

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = r"C:\Users\PC-LAB1\Desktop\llm\day09_sirah_rag"

INDEX_PATH = os.path.join(
    PROJECT_DIR,
    "05_vector_store",
    "sirah_faiss.index"
)

METADATA_PATH = os.path.join(
    PROJECT_DIR,
    "05_vector_store",
    "sirah_metadata.json"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "06_results"
)

OUTPUT_JSON = os.path.join(
    OUTPUT_DIR,
    "hybrid_retrieval_results.json"
)


# ============================================================
# MODELS
# ============================================================

EMBEDDING_MODEL = (
    "sayed0am/arabic-english-bge-m3"
)

RERANKER_MODEL = (
    "BAAI/bge-reranker-v2-m3"
)


# ============================================================
# RETRIEVAL SETTINGS
# ============================================================

DENSE_K = 40
BM25_K = 40

RRF_K = 60

RERANK_K = 25

FINAL_K = 5


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# LOAD FAISS
# ============================================================

print("=" * 70)
print("HYBRID ARABIC RETRIEVAL")
print("=" * 70)

index = faiss.read_index(
    INDEX_PATH
)


# ============================================================
# LOAD METADATA
# ============================================================

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    metadata = json.load(f)


chunks = metadata["chunks"]


print(
    f"\nChunks: {len(chunks):,}"
)

print(
    f"Vectors: {index.ntotal:,}"
)


# ============================================================
# ARABIC NORMALIZATION FOR BM25
# ============================================================

ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670]"
)

NON_WORD = re.compile(
    r"[^\w\u0600-\u06FF]+",
    re.UNICODE
)


def normalize_arabic(text):

    text = text.lower()

    text = text.replace(
        "أ",
        "ا"
    )

    text = text.replace(
        "إ",
        "ا"
    )

    text = text.replace(
        "آ",
        "ا"
    )

    text = text.replace(
        "ٱ",
        "ا"
    )

    text = text.replace(
        "ى",
        "ي"
    )

    text = text.replace(
        "ـ",
        ""
    )

    text = ARABIC_DIACRITICS.sub(
        "",
        text
    )

    text = NON_WORD.sub(
        " ",
        text
    )

    return text.strip()


def tokenize(text):

    normalized = normalize_arabic(
        text
    )

    return normalized.split()


# ============================================================
# BUILD BM25 STATISTICS
# ============================================================

print(
    "\nBuilding BM25 index..."
)


tokenized_documents = []

document_lengths = []

document_frequency = {}


for chunk in chunks:

    tokens = tokenize(
        chunk["text"]
    )

    tokenized_documents.append(
        tokens
    )

    document_lengths.append(
        len(tokens)
    )

    seen = set(tokens)

    for token in seen:

        document_frequency[token] = (
            document_frequency.get(
                token,
                0
            )
            + 1
        )


num_documents = len(
    tokenized_documents
)

average_document_length = (
    sum(document_lengths)
    / num_documents
    if num_documents
    else 0
)


# ============================================================
# BM25
# ============================================================

BM25_K1 = 1.5
BM25_B = 0.75


def bm25_scores(query):

    query_tokens = tokenize(
        query
    )

    query_term_set = set(
        query_tokens
    )

    scores = np.zeros(
        num_documents,
        dtype=np.float32
    )


    for token in query_term_set:

        if token not in document_frequency:
            continue

        df = document_frequency[
            token
        ]


        idf = math.log(
            1
            +
            (
                num_documents
                - df
                + 0.5
            )
            /
            (
                df
                + 0.5
            )
        )


        for doc_id, tokens in enumerate(
            tokenized_documents
        ):

            term_frequency = tokens.count(
                token
            )

            if term_frequency == 0:
                continue

            document_length = (
                document_lengths[doc_id]
            )

            denominator = (
                term_frequency
                +
                BM25_K1
                *
                (
                    1
                    -
                    BM25_B
                    +
                    BM25_B
                    *
                    document_length
                    /
                    average_document_length
                )
            )

            score = (
                idf
                *
                term_frequency
                *
                (BM25_K1 + 1)
                /
                denominator
            )

            scores[doc_id] += score


    return scores


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print(
    "\nLoading BGE-M3..."
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)


# ============================================================
# LOAD RERANKER
# ============================================================

print(
    "\nLoading reranker..."
)

reranker = CrossEncoder(
    RERANKER_MODEL
)


# ============================================================
# QUESTION
# ============================================================

question = input(
    "\nأدخل سؤالك بالعربية:\n> "
).strip()


if not question:

    raise ValueError(
        "Question cannot be empty."
    )


# ============================================================
# DENSE SEARCH
# ============================================================

print(
    "\nRunning dense retrieval..."
)

query_embedding = embedding_model.encode(
    [question],
    normalize_embeddings=True
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)


dense_scores, dense_indices = index.search(
    query_embedding,
    min(
        DENSE_K,
        index.ntotal
    )
)


# ============================================================
# BM25 SEARCH
# ============================================================

print(
    "Running BM25 retrieval..."
)

sparse_scores = bm25_scores(
    question
)

bm25_indices = np.argsort(
    sparse_scores
)[::-1][
    :min(
        BM25_K,
        len(chunks)
    )
]


# ============================================================
# RECIPROCAL RANK FUSION
# ============================================================

print(
    "Combining retrieval results..."
)

rrf_scores = {}


for rank, idx in enumerate(
    dense_indices[0],
    start=1
):

    idx = int(idx)

    rrf_scores[idx] = (
        rrf_scores.get(
            idx,
            0
        )
        +
        1.0
        /
        (
            RRF_K
            +
            rank
        )
    )


for rank, idx in enumerate(
    bm25_indices,
    start=1
):

    idx = int(idx)

    rrf_scores[idx] = (
        rrf_scores.get(
            idx,
            0
        )
        +
        1.0
        /
        (
            RRF_K
            +
            rank
        )
    )


candidate_indices = [
    idx
    for idx, score in sorted(
        rrf_scores.items(),
        key=lambda item:
            item[1],
        reverse=True
    )[:RERANK_K]
]


# ============================================================
# RERANK
# ============================================================

print(
    f"\nReranking {len(candidate_indices)} candidates..."
)


rerank_pairs = []

for idx in candidate_indices:

    rerank_pairs.append(
        [
            question,
            chunks[idx]["text"]
        ]
    )


rerank_scores = reranker.predict(
    rerank_pairs,
    batch_size=8,
    show_progress_bar=True
)


reranked = []


for idx, score in zip(
    candidate_indices,
    rerank_scores
):

    reranked.append(
        {
            "chunk_index":
                int(idx),

            "rerank_score":
                float(score),

            "rrf_score":
                float(
                    rrf_scores[idx]
                ),

            "bm25_score":
                float(
                    sparse_scores[idx]
                ),

            "dense_score":
                float(
                    dense_scores[0][
                        list(
                            dense_indices[0]
                        ).index(idx)
                    ]
                    if idx
                    in list(
                        dense_indices[0]
                    )
                    else 0.0
                ),

            "chunk":
                chunks[idx]
        }
    )


# ============================================================
# SORT BY RERANK SCORE
# ============================================================

reranked.sort(
    key=lambda item:
        item["rerank_score"],
    reverse=True
)


final_results = []

for rank, item in enumerate(
    reranked[:FINAL_K],
    start=1
):

    result = {

        "rank":
            rank,

        "chunk_id":
            item["chunk"]["chunk_id"],

        "page_start":
            item["chunk"]["page_start"],

        "page_end":
            item["chunk"]["page_end"],

        "word_count":
            item["chunk"]["word_count"],

        "rerank_score":
            item["rerank_score"],

        "rrf_score":
            item["rrf_score"],

        "bm25_score":
            item["bm25_score"],

        "dense_score":
            item["dense_score"],

        "text":
            item["chunk"]["text"]
    }

    final_results.append(
        result
    )


# ============================================================
# DISPLAY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RETRIEVED CHUNKS")
print("=" * 70)


for result in final_results:

    print("\n")

    print(
        f"RANK {result['rank']}"
    )

    print(
        f"Chunk      : "
        f"{result['chunk_id']}"
    )

    print(
        f"Pages      : "
        f"{result['page_start']}-"
        f"{result['page_end']}"
    )

    print(
        f"Rerank     : "
        f"{result['rerank_score']:.4f}"
    )

    print(
        f"RRF        : "
        f"{result['rrf_score']:.4f}"
    )

    print(
        f"BM25       : "
        f"{result['bm25_score']:.4f}"
    )

    print(
        f"Dense      : "
        f"{result['dense_score']:.4f}"
    )

    print("-" * 70)

    print(
        result["text"]
    )


# ============================================================
# SAVE
# ============================================================

output_data = {

    "question":
        question,

    "embedding_model":
        EMBEDDING_MODEL,

    "reranker_model":
        RERANKER_MODEL,

    "dense_k":
        DENSE_K,

    "bm25_k":
        BM25_K,

    "rerank_k":
        RERANK_K,

    "final_k":
        FINAL_K,

    "results":
        final_results
}


with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\n")
print("=" * 70)
print("HYBRID RETRIEVAL COMPLETE")
print("=" * 70)

print(
    f"\nSaved to:\n{OUTPUT_JSON}"
)